# 09. Регуляризация итоговой OLS-модели

Цель: добавить **Ridge / Lasso / Elastic Net** к финальной спецификации `M_fe_back` и сравнить качество на test.

In [ ]:
import sys, pathlib, json as _json, pickle
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from helpers import (set_plot_style, load_data, impute_median, make_design,
                     backward_elimination, derived_features, NUM_COLS_FULL, CAT_FEATURES)
import numpy as np, pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

set_plot_style()
df, df_train, df_test = load_data()
df_train = impute_median(df_train, NUM_COLS_FULL)
df_test  = df_test.copy()
for c in NUM_COLS_FULL:
    df_test[c] = df_test[c].fillna(df_train[c].median()).astype(float)

with open('selected_features.json') as fh:
    KEEP = _json.load(fh)['keep']

## 1. Восстановление итоговой OLS (`M_fe_back`)

In [ ]:
# train FE
df_train = derived_features(df_train, center=True)

# test FE: центрирование age/mileage по train-средним
df_test['hp_per_kg']      = df_test['engine_power_hp'] / df_test['weight_curb']
df_test['hp_per_liter']   = df_test['engine_power_hp'] / df_test['engine_volume']
df_test['ln_weight_curb'] = np.log(df_test['weight_curb'].clip(lower=1))
df_test['ln_mileage']     = np.log1p(df_test['mileage'])
df_test['age_c']          = df_test['age'] - df_train['age'].mean()
df_test['mileage_c']      = df_test['mileage'] - df_train['mileage'].mean()
df_test['age_c_sq']       = df_test['age_c'] ** 2
df_test['mileage_c_sq']   = (df_test['mileage_c'] / 1e5) ** 2
df_test['engine_x_age']   = df_test['engine_volume'] * df_test['age_c']

fe_num_base = [c for c in KEEP if c not in ('age', 'mileage', 'weight_curb')]
fe_num_base += ['age_c', 'mileage_c', 'ln_weight_curb']
fe_num = fe_num_base + ['hp_per_kg', 'hp_per_liter',
                         'age_c_sq', 'mileage_c_sq', 'ln_mileage',
                         'engine_x_age']

X_tr, _ = make_design(df_train, fe_num, CAT_FEATURES)
X_tr_c  = sm.add_constant(X_tr.astype(float), has_constant='add')
y_tr = df_train['price_log1p'].values
m_ols, X_tr_back, _ = backward_elimination(X_tr_c, y_tr, alpha=0.05)

X_te, _ = make_design(df_test, fe_num, CAT_FEATURES)
X_te_c  = sm.add_constant(X_te.astype(float), has_constant='add')
X_te_back = X_te_c.reindex(columns=X_tr_back.columns, fill_value=0.0)
y_te = df_test['price_log1p'].values

print(f'M_fe_back: n_train={len(y_tr)}, n_test={len(y_te)}, k={len(m_ols.params)}')
print(f'Train R²adj={m_ols.rsquared_adj:.4f}')

## 2. Ridge / Lasso / Elastic Net на той же итоговой спецификации

In [ ]:
feature_cols = [c for c in X_tr_back.columns if c != 'const']
X_train_reg = X_tr_back[feature_cols].copy()
X_test_reg  = X_te_back[feature_cols].copy()

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_reg)
X_test_sc  = scaler.transform(X_test_reg)

alphas = np.logspace(-4, 4, 80)
ridge = RidgeCV(alphas=alphas, cv=5).fit(X_train_sc, y_tr)
lasso = LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=50000).fit(X_train_sc, y_tr)
enet  = ElasticNetCV(alphas=alphas, l1_ratio=[0.1,0.3,0.5,0.7,0.9,0.95,0.99],
                     cv=5, random_state=42, max_iter=50000).fit(X_train_sc, y_tr)

print(f'Ridge alpha={ridge.alpha_:.6f}')
print(f'Lasso alpha={lasso.alpha_:.6f}')
print(f'ElasticNet alpha={enet.alpha_:.6f}, l1_ratio={enet.l1_ratio_:.2f}')

## 3. Сравнение метрик на test

In [ ]:
def eval_metrics(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return {
        'R² (log)': r2_score(y_true_log, y_pred_log),
        'MAE (log)': mean_absolute_error(y_true_log, y_pred_log),
        'RMSE (log)': mean_squared_error(y_true_log, y_pred_log) ** 0.5,
        'MAE (₽)': mean_absolute_error(y_true, y_pred),
        'RMSE (₽)': mean_squared_error(y_true, y_pred) ** 0.5,
        'MAPE (%)': np.mean(np.abs((y_pred - y_true) / np.clip(y_true, 1, None))) * 100,
    }

pred = {
    'OLS (M_fe_back)': np.asarray(m_ols.predict(X_te_back)),
    'RidgeCV': ridge.predict(X_test_sc),
    'LassoCV': lasso.predict(X_test_sc),
    'ElasticNetCV': enet.predict(X_test_sc),
}

rows = []
for name, yhat in pred.items():
    r = eval_metrics(y_te, yhat)
    r['модель'] = name
    rows.append(r)

res = pd.DataFrame(rows).set_index('модель')
res = res[['R² (log)','MAE (log)','RMSE (log)','MAE (₽)','RMSE (₽)','MAPE (%)']]
res = res.sort_values('R² (log)', ascending=False)
res.style.format({
    'R² (log)': '{:.4f}',
    'MAE (log)': '{:.4f}',
    'RMSE (log)': '{:.4f}',
    'MAE (₽)': '{:,.0f}',
    'RMSE (₽)': '{:,.0f}',
    'MAPE (%)': '{:.2f}',
})

## 4. Сколько коэффициентов занулил Lasso

In [ ]:
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'OLS β': m_ols.params.drop('const').reindex(feature_cols).values,
    'Ridge β': ridge.coef_,
    'Lasso β': lasso.coef_,
    'ElasticNet β': enet.coef_,
})

n_zero_lasso = int((np.abs(lasso.coef_) < 1e-10).sum())
n_zero_enet  = int((np.abs(enet.coef_) < 1e-10).sum())
print(f'Lasso занулил: {n_zero_lasso} из {len(feature_cols)} коэффициентов')
print(f'ElasticNet занулил: {n_zero_enet} из {len(feature_cols)} коэффициентов')

coef_df.assign(abs_lasso=np.abs(coef_df['Lasso β']))\
      .sort_values('abs_lasso')\
      .head(15)

## 5. Регуляризация в сегментации (vehicle_state)

Сравниваем `OLS / Ridge / Lasso / ElasticNet` в сегментном роутинге: 
- `на ходу` → модель сегмента `run`
- `битый / не на ходу` → модель сегмента `brk`


In [ ]:
# --- helpers для сегментации
def _segment_prepare(df_sub, age_mu, mil_mu):
    d = df_sub.copy()
    d['hp_per_kg']      = d['engine_power_hp'] / d['weight_curb']
    d['hp_per_liter']   = d['engine_power_hp'] / d['engine_volume']
    d['ln_weight_curb'] = np.log(d['weight_curb'].clip(lower=1))
    d['ln_mileage']     = np.log1p(d['mileage'])
    d['age_c']          = d['age'] - age_mu
    d['mileage_c']      = d['mileage'] - mil_mu
    d['age_c_sq']       = d['age_c'] ** 2
    d['mileage_c_sq']   = (d['mileage_c'] / 1e5) ** 2
    d['engine_x_age']   = d['engine_volume'] * d['age_c']
    return d

def _fit_segment_models(df_seg_train):
    age_mu = df_seg_train['age'].mean()
    mil_mu = df_seg_train['mileage'].mean()
    d_tr = _segment_prepare(df_seg_train, age_mu, mil_mu)

    X_tr_full, _ = make_design(d_tr, fe_num, CAT_FEATURES)
    X_tr_full_c  = sm.add_constant(X_tr_full.astype(float), has_constant='add')
    y_tr_seg = d_tr['price_log1p'].values

    m_ols_seg, X_tr_back_seg, _ = backward_elimination(X_tr_full_c, y_tr_seg, alpha=0.05)

    feat_seg = [c for c in X_tr_back_seg.columns if c != 'const']
    scaler_seg = StandardScaler()
    X_tr_sc_seg = scaler_seg.fit_transform(X_tr_back_seg[feat_seg])

    alphas = np.logspace(-4, 4, 80)
    ridge_seg = RidgeCV(alphas=alphas, cv=5).fit(X_tr_sc_seg, y_tr_seg)
    lasso_seg = LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=50000).fit(X_tr_sc_seg, y_tr_seg)
    enet_seg  = ElasticNetCV(alphas=alphas, l1_ratio=[0.1,0.3,0.5,0.7,0.9,0.95,0.99],
                             cv=5, random_state=42, max_iter=50000).fit(X_tr_sc_seg, y_tr_seg)

    return {
        'age_mu': age_mu, 'mil_mu': mil_mu,
        'ols': m_ols_seg, 'cols': list(X_tr_back_seg.columns), 'feat': feat_seg,
        'scaler': scaler_seg, 'ridge': ridge_seg, 'lasso': lasso_seg, 'enet': enet_seg
    }

def _predict_segment(model_pack, df_seg_test, kind='ols'):
    d_te = _segment_prepare(df_seg_test, model_pack['age_mu'], model_pack['mil_mu'])
    X_te_full, _ = make_design(d_te, fe_num, CAT_FEATURES)
    X_te_full_c  = sm.add_constant(X_te_full.astype(float), has_constant='add')
    X_te_back = X_te_full_c.reindex(columns=model_pack['cols'], fill_value=0.0)

    if kind == 'ols':
        yhat = np.asarray(model_pack['ols'].predict(X_te_back))
    else:
        X_te_sc = model_pack['scaler'].transform(X_te_back[model_pack['feat']])
        if kind == 'ridge':
            yhat = model_pack['ridge'].predict(X_te_sc)
        elif kind == 'lasso':
            yhat = model_pack['lasso'].predict(X_te_sc)
        elif kind == 'enet':
            yhat = model_pack['enet'].predict(X_te_sc)
        else:
            raise ValueError('unknown kind')

    return d_te['price_log1p'].values, np.asarray(yhat)

# --- train сегментных моделей
tr_run = df_train[df_train['vehicle_state'] == 'на ходу'].reset_index(drop=True)
tr_brk = df_train[df_train['vehicle_state'] == 'битый / не на ходу'].reset_index(drop=True)
te_run = df_test[df_test['vehicle_state'] == 'на ходу'].reset_index(drop=True)
te_brk = df_test[df_test['vehicle_state'] == 'битый / не на ходу'].reset_index(drop=True)

pack_run = _fit_segment_models(tr_run)
pack_brk = _fit_segment_models(tr_brk)

def _routed_pred(kind):
    yr, yhr = _predict_segment(pack_run, te_run, kind=kind)
    yb, yhb = _predict_segment(pack_brk, te_brk, kind=kind)
    y_true = np.concatenate([yr, yb])
    y_pred = np.concatenate([yhr, yhb])
    return y_true, y_pred

rows_seg = []
for nm, kind in [('OLS seg (state)', 'ols'), ('RidgeCV seg (state)', 'ridge'),
                 ('LassoCV seg (state)', 'lasso'), ('ElasticNetCV seg (state)', 'enet')]:
    yt, yp = _routed_pred(kind)
    r = eval_metrics(yt, yp)
    r['модель'] = nm
    rows_seg.append(r)

seg_res = pd.DataFrame(rows_seg).set_index('модель')
seg_res = seg_res[['R² (log)','MAE (log)','RMSE (log)','MAE (₽)','RMSE (₽)','MAPE (%)']]
seg_res = seg_res.sort_values('R² (log)', ascending=False)

print(f"run: Ridge a={pack_run['ridge'].alpha_:.6f}, Lasso a={pack_run['lasso'].alpha_:.6f}, ENet a={pack_run['enet'].alpha_:.6f}, l1={pack_run['enet'].l1_ratio_:.2f}")
print(f"brk: Ridge a={pack_brk['ridge'].alpha_:.6f}, Lasso a={pack_brk['lasso'].alpha_:.6f}, ENet a={pack_brk['enet'].alpha_:.6f}, l1={pack_brk['enet'].l1_ratio_:.2f}")
seg_res.style.format({
    'R² (log)': '{:.4f}',
    'MAE (log)': '{:.4f}',
    'RMSE (log)': '{:.4f}',
    'MAE (₽)': '{:,.0f}',
    'RMSE (₽)': '{:,.0f}',
    'MAPE (%)': '{:.2f}',
})


## 6. Проверка гетероскедастичности в сегментной OLS

Диагностика делается отдельно внутри каждого сегмента (`на ходу`, `битые`) по train-остаткам.


In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan, het_white

def _hetero_table(pack, label):
    resid = np.asarray(pack['ols'].resid)
    exog = np.asarray(pack['ols'].model.exog)

    bp_lm, bp_p, _, _ = het_breuschpagan(resid, exog)

    # White может быть нестабилен при вырожденной матрице (много dummy в сегменте)
    try:
        w_lm, w_p, _, _ = het_white(resid, exog)
        w_lm_val = round(float(w_lm), 1)
        w_p_val = f'{w_p:.3g}'
        w_note = ''
    except Exception:
        w_lm_val = np.nan
        w_p_val = 'nan'
        w_note = 'white недоступен (rank issue)'

    return {
        'сегмент': label,
        'n_train': int(len(resid)),
        'BP LM': round(float(bp_lm), 1),
        'BP p-value': f'{bp_p:.3g}',
        'White LM': w_lm_val,
        'White p-value': w_p_val,
        'примечание': w_note,
        'α=0.05 (по BP)': 'отвергаем H₀' if bp_p < 0.05 else 'не отвергаем H₀',
    }

hetero_seg = pd.DataFrame([
    _hetero_table(pack_run, 'на ходу'),
    _hetero_table(pack_brk, 'битые'),
]).set_index('сегмент')
print('H₀: гомоскедастичность (постоянная дисперсия остатков).')
hetero_seg



## 7. Сохранение артефактов

Сохраняем модели и scaler для возможного использования в следующих блоках.

In [ ]:
with open('regularization_models.pkl', 'wb') as fh:
    pickle.dump({
        'feature_cols': feature_cols,
        'scaler': scaler,
        'ridge': ridge,
        'lasso': lasso,
        'elastic_net': enet,
        'ols_ref': m_ols,
    }, fh)
print('Сохранено: regularization_models.pkl')

## Выводы блока 9

- Регуляризация сравнивается **на той же финальной OLS-спецификации**.
- `Ridge` обычно даёт самый стабильный прирост/стабильность при мультиколлинеарности.
- `Lasso` и `Elastic Net` дополнительно делают сжатие/отбор коэффициентов.